In [37]:
import gymnasium as  gym
from gym import envs
import matplotlib.pyplot as plt
import numpy as np
import random
import pickle

In [38]:
all_envs = envs.registry.values()

env_ids = sorted([env.id for env in all_envs])
print(f"Total number of environments: {len(env_ids)}\n")

for i, eid in enumerate(env_ids[:100]):   # showing first 100 to avoid huge output
    print(f"{i+1}. {eid}")



Total number of environments: 44

1. Acrobot-v1
2. Ant-v2
3. Ant-v3
4. Ant-v4
5. BipedalWalker-v3
6. BipedalWalkerHardcore-v3
7. Blackjack-v1
8. CarRacing-v2
9. CartPole-v0
10. CartPole-v1
11. CliffWalking-v0
12. FrozenLake-v1
13. FrozenLake8x8-v1
14. HalfCheetah-v2
15. HalfCheetah-v3
16. HalfCheetah-v4
17. Hopper-v2
18. Hopper-v3
19. Hopper-v4
20. Humanoid-v2
21. Humanoid-v3
22. Humanoid-v4
23. HumanoidStandup-v2
24. HumanoidStandup-v4
25. InvertedDoublePendulum-v2
26. InvertedDoublePendulum-v4
27. InvertedPendulum-v2
28. InvertedPendulum-v4
29. LunarLander-v2
30. LunarLanderContinuous-v2
31. MountainCar-v0
32. MountainCarContinuous-v0
33. Pendulum-v1
34. Pusher-v2
35. Pusher-v4
36. Reacher-v2
37. Reacher-v4
38. Swimmer-v2
39. Swimmer-v3
40. Swimmer-v4
41. Taxi-v3
42. Walker2d-v2
43. Walker2d-v3
44. Walker2d-v4


In [39]:
env = gym.make("FrozenLake-v1", render_mode="ansi",is_slippery=False)
env.reset(seed=42)

(0, {'prob': 1})

In [40]:
print("\nInitial rendered environment:\n")
print(env.render())


Initial rendered environment:


SFFF
FHFH
FFFH
HFFG



In [41]:
env.step(1) 
env.step(1) 
env.step(2)

(9, 0, False, False, {'prob': 1.0})

In [42]:
print("--- STATE TO SAVE ---")
print(env.render())
current_pos = env.unwrapped.s
print(f"Current Position (State Index): {current_pos}")

--- STATE TO SAVE ---
  (Right)
SFFF
FHFH
FFFH
HFFG

Current Position (State Index): 9


In [43]:
print("Action Space:", env.action_space)
print("Action Space - number of actions:", env.action_space.n)

Action Space: Discrete(4)
Action Space - number of actions: 4


In [44]:
print("\nObservation Space:", env.observation_space)
print("Observation Space - number of states:", env.observation_space.n)


Observation Space: Discrete(16)
Observation Space - number of states: 16


In [45]:
snapshot = {
    "env_pos": env.unwrapped.s, # Specifically for FrozenLake, 's' is the state
    "np_rng": np.random.get_state(),
    "py_rng": random.getstate(),
    "env_rng": env.np_random.bit_generator.state # <--- THE FIX
}

# Save to file
with open("snapshot.pkl", "wb") as f:
    pickle.dump(snapshot, f)
print("\nSnapshot saved to 'snapshot.pkl'.")


Snapshot saved to 'snapshot.pkl'.


In [46]:
print("\n--- ALTERING STATE (Messing it up) ---")
env.reset() # Reset back to start
env.step(2) # Move right
print(env.render())
print(f"Current Position: {env.unwrapped.s}")
print("Environment is now different from the saved state.")


--- ALTERING STATE (Messing it up) ---
  (Right)
SFFF
FHFH
FFFH
HFFG

Current Position: 1
Environment is now different from the saved state.


In [47]:
#  Reload Snapshot
print("\n--- RESTORING STATE ---")
with open("snapshot.pkl", "rb") as f:
    loaded_snapshot = pickle.load(f)

    # Restore Random States
np.random.set_state(loaded_snapshot["np_rng"])
random.setstate(loaded_snapshot["py_rng"])

# Restore Environment RNG
# We assign the state back to the bit_generator
env.np_random.bit_generator.state = loaded_snapshot["env_rng"]

# Restore Environment Internal State
# For FrozenLake, we specifically restore 's' (position)
env.unwrapped.s = loaded_snapshot["env_pos"]
print(env.render())

print(f"Restored Position: {env.unwrapped.s}")

if env.unwrapped.s == current_pos:
    print("\nSUCCESS: State successfully restored!")
else:
    print("\nFAILURE: State mismatch.")

env.close()


--- RESTORING STATE ---
  (Right)
SFFF
FHFH
FFFH
HFFG

Restored Position: 9

SUCCESS: State successfully restored!
